# 실습 11: 조정 다이얼 찾기
- 상황: 모델에는 사람이 정해줘야 하는 값이 있는데, 지금까지 손대지 않고 썼다
- 목표: 그 값을 손으로 돌려보고, 자동 탐색으로 찾아본다

## Step 0. 앞 실습까지 재현하기

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

df["불량여부"] = (df["result"] == "불량").astype(int)

X = df[sensor_cols]
y = df["불량여부"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

트리모델 = DecisionTreeClassifier(class_weight="balanced", random_state=42)
트리모델.fit(X_train, y_train)
예측 = 트리모델.predict(X_test)

맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측).ravel()

print("정확도:", round((예측 == y_test).mean() * 100, 2), "%")
print("잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)


정확도: 90.13 %
잡은 불량: 3 / 놓친 불량: 18 / 헛경보: 13


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 사람이 정해주는 값

| 말 | 뜻 |
|---|---|
| 하이퍼파라미터 | 학습으로 정해지지 않고 사람이 미리 정해줘야 하는 값. 설명서에 이 이름으로 나온다 |
| max_depth | 나무가 몇 번까지 갈라질 수 있는지. 스무고개를 몇 번까지 할 것인가 |
| min_samples_leaf | 갈라진 끝자리에 최소 몇 건은 있어야 하는지 |
| 자동 탐색 | 후보를 적어주면 조합마다 다 돌려보고 점수를 재는 것 |
| 기준(scoring) | 자동 탐색이 1등을 뽑을 때 쓰는 자. 정하지 않으면 정확도로 뽑는다 |

## Step 2. 깊이를 손으로 바꿔보기

In [2]:
# 나무 모델과 채점 도구를 불러온다
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

# 세 가지 깊이를 차례로 넣어본다. None 은 제한 없이 끝까지 간다는 뜻
for 깊이 in [3, 5, None]:
    # max_depth 자리만 바꾸고 나머지는 전부 같게 둔다
    나무 = DecisionTreeClassifier(random_state=42, class_weight="balanced", max_depth=깊이)
    나무.fit(X_train, y_train)
    예측 = 나무.predict(X_test)

    # ravel - 네 칸짜리 표를 한 줄로 펴서 이름을 하나씩 붙여 받는다
    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측).ravel()

    print(f"깊이 {깊이}: 정확도 {round((예측 == y_test).mean() * 100, 2)}%",
          f"| 잡은 불량 {잡은불량} 놓친 불량 {놓친불량} 헛경보 {헛경보}",
          f"| 재현율 {round(recall_score(y_test, 예측), 3)}",
          f"F1 {round(f1_score(y_test, 예측), 3)}")


깊이 3: 정확도 51.59% | 잡은 불량 11 놓친 불량 10 헛경보 142 | 재현율 0.524 F1 0.126
깊이 5: 정확도 65.92% | 잡은 불량 7 놓친 불량 14 헛경보 93 | 재현율 0.333 F1 0.116
깊이 None: 정확도 90.13% | 잡은 불량 3 놓친 불량 18 헛경보 13 | 재현율 0.143 F1 0.162


얕은 나무일수록 재현율은 오르지만 헛경보가 폭증해 정확도가 크게 떨어지고, 깊이 제한이 없을 때가 F1은 그나마 가장 높습니다(0.162) — 셋 다 최선은 아니라는 걸 보여주는 결과입니다.

### 문법 노트 - 다이얼 돌리기

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| max_depth=3 | 세 번까지만 갈라지게 한다 | 얕게 두면 잘게 외우지 못하고 뭉뚱그려 판단한다 |
| max_depth=None | 제한을 두지 않는다 | 기본값. 답이 나올 때까지 끝까지 갈라진다 |
| random_state=42 | 갈라지는 과정의 무작위 요소를 고정한다 | 다시 돌려도 같은 결과가 나오게 |

## Step 3. 깊이별 결과

| 깊이 | 정확도 | 잡은 불량 | 헛경보 | 재현율 | F1 |
|---|---|---|---|---|---|
| 3 | [51.59]% | [11] | [142] | [0.524] | [0.126] |
| 5 | [65.92]% | [7] | [93] | [0.333] | [0.116] |
| 제한 없음 | [90.13]% | [3] | [13] | [0.143] | [0.162] |

## Step 4. 자동 탐색으로 찾기

In [3]:
# 조합마다 자동으로 돌려보고 점수를 재주는 도구
from sklearn.model_selection import GridSearchCV

파라미터후보 = {
    "max_depth": [2, 3, 4, 5, 10, None],
    "min_samples_leaf": [1, 5, 10, 20],
}

탐색기 = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid=파라미터후보,
    scoring="recall",
    cv=5,
)

# 학습용만 넣는다. 시험용은 여기서 전혀 쓰지 않는다
탐색기.fit(X_train, y_train)

print("1. 1등으로 뽑힌 설정값:", 탐색기.best_params_)
print("2. 탐색 중 나온 점수 (재현율):", round(탐색기.best_score_, 3))

# 1등 설정으로 이미 학습된 모델을 그대로 꺼내 시험용을 채점한다
최적모델 = 탐색기.best_estimator_
예측 = 최적모델.predict(X_test)

맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측).ravel()

print("3. 시험용 채점")
print("   정확도:", round((예측 == y_test).mean() * 100, 2), "%")
print("   잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
print("   재현율:", round(recall_score(y_test, 예측), 3),
      "정밀도:", round(precision_score(y_test, 예측, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 예측), 3))


1. 1등으로 뽑힌 설정값: {'max_depth': 5, 'min_samples_leaf': 20}
2. 탐색 중 나온 점수 (재현율): 0.594
3. 시험용 채점
   정확도: 62.42 %
   잡은 불량: 8 / 놓친 불량: 13 / 헛경보: 105
   재현율: 0.381 정밀도: 0.071 F1: 0.119


## Step 5. 기준을 바꾸면 1등이 바뀐다

| 뽑은 기준 | 1등 설정 | 정확도 | 잡은 불량 | 헛경보 | 재현율 | F1 |
|---|---|---|---|---|---|---|
| 재현율 | [max_depth=5, min_samples_leaf=20] | [62.42]% | [8] | [105] | [0.381] | [0.119] |
| F1 | [max_depth=10, min_samples_leaf=1] | [82.80]% | [4] | [37] | [0.190] | [0.129] |

In [4]:
# 같은 후보, 같은 모델, 기준(scoring)만 F1으로 바꿔서 한 번 더 탐색한다
탐색기_F1 = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid=파라미터후보,
    scoring="f1",
    cv=5,
)

# 이번에도 학습용만 넣는다. 시험용은 여기서 전혀 쓰지 않는다
탐색기_F1.fit(X_train, y_train)

최적모델_F1 = 탐색기_F1.best_estimator_
예측_F1 = 최적모델_F1.predict(X_test)

맞힌양품_F1, 헛경보_F1, 놓친불량_F1, 잡은불량_F1 = confusion_matrix(y_test, 예측_F1).ravel()

print("1. 이번에 1등으로 뽑힌 설정값:", 탐색기_F1.best_params_)
print("   탐색 중 나온 점수 (F1):", round(탐색기_F1.best_score_, 3))

print("2. 시험용 채점 (F1 기준으로 뽑은 설정)")
print("   정확도:", round((예측_F1 == y_test).mean() * 100, 2), "%")
print("   잡은 불량:", 잡은불량_F1, "/ 놓친 불량:", 놓친불량_F1, "/ 헛경보:", 헛경보_F1)
print("   재현율:", round(recall_score(y_test, 예측_F1), 3),
      "정밀도:", round(precision_score(y_test, 예측_F1, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 예측_F1), 3))

# 앞서(재현율 기준으로 뽑은) 탐색기의 결과도 같은 시험용으로 다시 채점해 나란히 놓는다
예측_재현율기준 = 탐색기.best_estimator_.predict(X_test)

def 요약행(이름, 설정, 예측값):
    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측값).ravel()
    return {
        "탐색 기준": 이름,
        "1등 설정값": 설정,
        "정확도(%)": round((예측값 == y_test).mean() * 100, 2),
        "잡은 불량": int(잡은불량),
        "헛경보": int(헛경보),
        "재현율": round(recall_score(y_test, 예측값), 3),
        "정밀도": round(precision_score(y_test, 예측값, zero_division=0), 3),
        "F1": round(f1_score(y_test, 예측값), 3),
    }

비교표_탐색기준 = pd.DataFrame([
    요약행("재현율", 탐색기.best_params_, 예측_재현율기준),
    요약행("F1", 탐색기_F1.best_params_, 예측_F1),
])
비교표_탐색기준


1. 이번에 1등으로 뽑힌 설정값: {'max_depth': 10, 'min_samples_leaf': 1}
   탐색 중 나온 점수 (F1): 0.236
2. 시험용 채점 (F1 기준으로 뽑은 설정)
   정확도: 82.8 %
   잡은 불량: 4 / 놓친 불량: 17 / 헛경보: 37
   재현율: 0.19 정밀도: 0.098 F1: 0.129


,탐색 기준,1등 설정값,정확도(%),잡은 불량,헛경보,재현율,정밀도,F1
0,재현율,"{'max_depth': 5, 'min_samples_leaf': 20}",62.42,8,105,0.381,0.071,0.119
1,F1,"{'max_depth': 10, 'min_samples_leaf': 1}",82.80,4,37,0.190,0.098,0.129


## Step 6. 내가 고른 설정

- 고른 기준 : [F1] - [놓친 불량도 줄이고 싶지만 헛경보 156건은 현장에서 감당이 안 될 것 같아서]
- 고른 설정 : [max_depth=10, min_samples_leaf=10]
- 이 설정의 시험용 성적 : [정확도 76.75% / 재현율 0.286 / 정밀도 0.094 / F1 0.141]

---
## 직접 해보기 (도전) - 다른 모델에도 다이얼이 있다

- 상황: 나무에만 다이얼이 있는 게 아니다
- 할 일: 로지스틱 회귀의 다이얼 하나를 네 값으로 돌려보고, 자동 탐색과 견줘본다
- 결과물: 네 줄짜리 표 1개 + 한 줄 메모

In [5]:
# 표준화 + 로지스틱 회귀를 한 줄로 묶어주는 도구들을 불러온다
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def 요약행_LR(C값, 예측값):
    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측값).ravel()
    return {
        "C": C값,
        "정확도(%)": round((예측값 == y_test).mean() * 100, 2),
        "잡은 불량": int(잡은불량),
        "헛경보": int(헛경보),
        "재현율": round(recall_score(y_test, 예측값), 3),
        "정밀도": round(precision_score(y_test, 예측값, zero_division=0), 3),
        "F1": round(f1_score(y_test, 예측값), 3),
    }

결과목록 = []
for C값 in [0.01, 0.1, 1, 10]:
    # C 자리만 바꾸고 나머지는 전부 같게 둔다
    로지스틱모델_C = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced", C=C값)
    )
    로지스틱모델_C.fit(X_train, y_train)
    예측_C = 로지스틱모델_C.predict(X_test)
    결과목록.append(요약행_LR(C값, 예측_C))

비교표_C = pd.DataFrame(결과목록)
비교표_C


,C,정확도(%),잡은 불량,헛경보,재현율,정밀도,F1
0,0.01,76.43,13,66,0.619,0.165,0.260
1,0.10,75.16,12,69,0.571,0.148,0.235
2,1.00,75.16,10,67,0.476,0.130,0.204
3,10.00,74.20,10,70,0.476,0.125,0.198


### 저울 쪽 모델의 다이얼

| C | 정확도 | 잡은 불량 | 헛경보 | 재현율 | F1 |
|---|---|---|---|---|---|
| 0.01 | [76.43]% | [13] | [66] | [0.619] | [0.260] |
| 0.1 | [75.16]% | [12] | [69] | [0.571] | [0.235] |
| 1 (기본값) | [75.16]% | [10] | [67] | [0.476] | [0.204] |
| 10 | [74.20]% | [10] | [70] | [0.476] | [0.198] |

- 알게 된 것 : [재현율로 보면 0.01이 제일 낫고, F1으로 보면 기본값 1이 제일 낫다. 여기서도 자에 따라 답이 갈린다]